# Task 3 — Gender and Usage Classification

**Status:** planning scaffold only. There is no training code, selected model,
selected metric, fake result, or final claim in this notebook.

**Owner:** TODO(owner)


## Colab setup — repository and teacher data

Run these cells at the start of each fresh Colab runtime. They are safe to rerun.
The repository supplies `data/processed`, including the one allowed split. The Drive
archive supplies only `data/raw/teacher`. Images are extracted to Colab's local disk
because training directly from Drive is much slower.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"


def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    return subprocess.run(command, cwd=cwd, check=True)


In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("This setup cell must run in Google Colab.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)

if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=REPO_DIR, text=True
    ).strip()
    if dirty:
        print("Local repository changes found; skipped the automatic fast-forward update.")
    else:
        run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR]
    )

print(f"Repository ready: {REPO_DIR} ({BRANCH})")


In [ ]:
if not DATA_ZIP.is_file():
    raise FileNotFoundError(
        f"Dataset archive not found at {DATA_ZIP}. Check the Drive folder and file name."
    )

teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)

with zipfile.ZipFile(DATA_ZIP) as archive:
    names = archive.namelist()
    unsafe_names = [
        name for name in names if Path(name).is_absolute() or ".." in Path(name).parts
    ]
    if unsafe_names:
        raise RuntimeError("The dataset archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and name.lower().endswith((".jpg", ".jpeg"))
        for name in names
    )
    if expected_images == 0:
        raise RuntimeError("The archive does not contain teacher images in the expected layout.")

    current_images = sum(1 for path in teacher_dir.rglob("*") if path.suffix.lower() in {".jpg", ".jpeg"})
    needs_extract = current_images != expected_images or not all(path.is_file() for path in required_files)
    if needs_extract:
        print(f"Extracting {expected_images:,} teacher images from Drive...")
        archive.extractall(REPO_DIR)
    else:
        print("Teacher data is already extracted; skipping.")

actual_images = sum(1 for path in teacher_dir.rglob("*") if path.suffix.lower() in {".jpg", ".jpeg"})
missing_files = [str(path) for path in required_files if not path.is_file()]
if actual_images != expected_images or missing_files:
    raise RuntimeError(
        f"Dataset check failed: expected {expected_images:,} images, found {actual_images:,}; "
        f"missing files: {missing_files}"
    )

print(f"Teacher data ready: {actual_images:,} images in {teacher_dir}")


In [ ]:
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

for output_dir in (
    DRIVE_TASK_DIR / "checkpoints",
    DRIVE_TASK_DIR / "logs",
    DRIVE_TASK_DIR / "results",
):
    output_dir.mkdir(parents=True, exist_ok=True)

from fashion.data import get_samples, load_label_maps, load_splits

splits = load_splits()
label_maps = load_label_maps()
gender_development = get_samples(splits, partition="development", target="gender")
usage_development = get_samples(splits, partition="development", target="usage")

print("Task 3 data is ready.")
print(f"Gender development rows: {len(gender_development):,}")
print(f"Usage development rows:  {len(usage_development):,}")
print(f"Persistent outputs:       {DRIVE_TASK_DIR}")
splits.groupby("partition", observed=True).size().rename("rows").to_frame()


## 1. Task contract

- Decision this output supports: TODO(owner)
- Prediction/retrieval unit: TODO(owner)
- Required output format: TODO(owner; verify against the assignment specification)
- In-scope and out-of-scope behaviour: TODO(owner)
- Known assumptions and risks: TODO(owner)


## 2. Data and task output

- Shared manifest: `data/processed/splits.csv`
- Task output: separate `gender` and `usage` outputs; preserve the required submission columns
- Image source boundary: teacher images only
- Missing-label rule for this task: TODO(owner; use the matching `has_<target>_label` mask)
- Product/family grouping fields used for leakage checks: TODO(owner)


## 3. Development-validation strategy

**CV mode: TODO(owner)**

Choose and record one mode before experiments:

- one fixed validation fold; or
- all five precomputed folds.

Give the reason, compute budget, and comparison rule. Do not run several folds
and report only the best-looking one.


## 4. Fixed-fold declaration

**Validation fold if fixed mode: TODO(owner)**

If all five folds are used, write `not applicable — all folds` and define how
fold results will be aggregated before seeing them.


## 5. Task-specific preprocessing and leakage rules

- Inputs available at prediction time: TODO(owner)
- Task transform pipeline: TODO(owner)
- Values learned from data: TODO(owner)
- Fit boundary: training folds of the current round only
- Validation application rule: TODO(owner; transform only, never refit)
- Holdout rule: sealed until Notebook 06 after every choice is frozen

If mean/std, rebalancing values, sampling rules, or feature selection are used,
fit them again inside each round's training rows. Do not copy a value fitted on
all development into a fold experiment.


## 6. Preprocessing comparisons to run

Define the alternatives that answer a real question for this task.

| Comparison ID | Question | Alternative A | Alternative B | Controlled variables | Evidence needed |
|---|---|---|---|---|---|
| TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) |

Do not choose an alternative in this scaffold.


## 7. Hypotheses and baseline

- Baseline purpose and implementation: TODO(owner)
- Hypothesis 1 and expected observation: TODO(owner)
- Hypothesis 2 and expected observation: TODO(owner)
- Separate versus shared representation question: TODO(owner)
- Label-mask and loss-weight comparison needed: TODO(owner)
- Failure condition that would reject each hypothesis: TODO(owner)

A baseline is a comparison anchor, not a preselected winner.


## 8. Candidate model comparisons

Define candidate families only after checking the assignment constraints.

| Candidate ID | Why include it | Capacity/complexity control | Scratch-training compliance | Expected trade-off |
|---|---|---|---|---|
| TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) |

No candidate is selected in advance.


## 9. Metric selection and freeze

- Primary development metric for `gender`: TODO(owner)
- Primary development metric for `usage`: TODO(owner)
- Why it matches the task decision: TODO(owner)
- Secondary diagnostic measures: TODO(owner)
- Aggregation across classes/outputs/folds: TODO(owner)
- Freeze timestamp or decision-log entry: TODO(owner)

Choose and document the metric before comparing final candidates. Do not edit the
metric because a later result looks inconvenient.


## 10. Experiment matrix

| Run intent | CV mode/fold | Preprocessing ID | Candidate ID | Controlled seed/budget | Question answered |
|---|---|---|---|---|---|
| TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) |

Keep the matrix broad enough to justify choices, but small enough that each run
receives error and cost analysis.


## 11. Run registry

Every training/evaluation run must append through `fashion.train.registry` to
`results/runs.csv`.

- Run IDs: TODO(owner after execution)
- Checkpoint paths: TODO(owner after execution)
- Configuration hashes: TODO(owner after execution)
- Evidence tables/figures linked to run IDs: TODO(owner after execution)

Do not hand-type final comparison numbers into the report.


## 12. Error analysis

- Define useful error slices before viewing results: TODO(owner)
- Inspect representative successes and failures with IDs: TODO(owner)
- Check rare/ambiguous groups and family effects: TODO(owner)
- Check negative transfer between gender and usage: TODO(owner)
- Separate data limitations from model limitations: TODO(owner)
- Record unexpected failure modes honestly: TODO(owner)


## 13. Robustness and efficiency

- Robustness questions and controlled tests: TODO(owner)
- Runtime and hardware measurement rule: TODO(owner)
- Memory/storage or index cost: TODO(owner)
- Stability across seeds/folds where applicable: TODO(owner)
- Practical deployment limitation: TODO(owner)


## 14. Decision log

| Decision | Evidence considered | Choice | Rejected alternatives | Limitation | Date/owner |
|---|---|---|---|---|---|
| TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) |

Record choices as they are made. Do not rewrite history after holdout access.


## 15. Handoff to final evaluation

Before Notebook 06, provide:

- frozen winning run ID(s): TODO(owner)
- frozen preprocessing configuration: TODO(owner)
- frozen metric definition and CV evidence: TODO(owner)
- refit procedure for all development: TODO(owner)
- expected final checkpoint/output path: TODO(owner)
- unresolved risks and honest limitations: TODO(owner)

**Handoff status: NOT READY — owner must complete every item above.**
